## **Aim**
To implement a program that generates a digital event timeline using file timestamps and system log records.

## **Algorithm**
**Step 1:** Import `os`, `json`, `datetime`, `glob`, and `stat` libraries.

**Step 2:** Create simulated data sources:
   - File system metadata (creation, modification, access times)
   - System logs (auth log, process execution log, USB log)
   - Network logs (connection events)

**Step 3:** Define a function `collect_file_timestamps(directory)` to gather timestamps from all files.

**Step 4:** Define a function `parse_log_timestamps(log_files)` to extract timestamped events from log files.

**Step 5:** Normalize all events to a common format: timestamp, source, event_type, description, details.

**Step 6:** Sort all events chronologically.

**Step 7:** Apply filters (time range, event types, keywords) and generate timeline output.

**Step 8:** Export timeline to JSON and human-readable text formats.

In [1]:
import os
import json
import glob
from datetime import datetime, timedelta
from collections import defaultdict
import stat

def collect_file_timestamps(directory, recursive=True):
    """Collect timestamps from files in directory"""
    events = []
    pattern = os.path.join(directory, "**" if recursive else "*")
    
    for filepath in glob.glob(pattern, recursive=recursive):
        if not os.path.isfile(filepath):
            continue
        try:
            st = os.stat(filepath)
            rel_path = os.path.relpath(filepath, directory)
            
            events.append({
                "timestamp": datetime.fromtimestamp(st.st_ctime).isoformat(),
                "source": "FILESYSTEM",
                "event_type": "FILE_CREATED",
                "description": f"File created: {rel_path}",
                "details": {"path": rel_path, "size": st.st_size, "inode": st.st_ino}
            })
            events.append({
                "timestamp": datetime.fromtimestamp(st.st_mtime).isoformat(),
                "source": "FILESYSTEM",
                "event_type": "FILE_MODIFIED",
                "description": f"File modified: {rel_path}",
                "details": {"path": rel_path, "size": st.st_size}
            })
            events.append({
                "timestamp": datetime.fromtimestamp(st.st_atime).isoformat(),
                "source": "FILESYSTEM",
                "event_type": "FILE_ACCESSED",
                "description": f"File accessed: {rel_path}",
                "details": {"path": rel_path}
            })
        except Exception:
            pass
    return events

def parse_auth_log(log_file):
    """Parse auth.log for login events"""
    events = []
    if not os.path.exists(log_file):
        return events
    
    import re
    ip_pattern = r'(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'
    
    with open(log_file, "r") as f:
        for line in f:
            # Parse timestamp (format: Aug 6 10:01:01)
            match = re.match(r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})', line)
            if not match:
                continue
            
            ts_str = match.group(1)
            try:
                # Assume current year
                timestamp = datetime.strptime(f"{datetime.now().year} {ts_str}", "%Y %b %d %H:%M:%S")
            except Exception:
                continue
            
            ip_match = re.search(ip_pattern, line)
            ip = ip_match.group(1) if ip_match else "unknown"
            
            if "Accepted password" in line:
                user_match = re.search(r'Accepted password for (\S+)', line)
                user = user_match.group(1) if user_match else "unknown"
                events.append({
                    "timestamp": timestamp.isoformat(),
                    "source": "AUTH_LOG",
                    "event_type": "LOGIN_SUCCESS",
                    "description": f"Successful login: {user} from {ip}",
                    "details": {"user": user, "ip": ip, "method": "password"}
                })
            elif "Failed password" in line:
                user_match = re.search(r'Failed password for (\S+)', line)
                user = user_match.group(1) if user_match else "unknown"
                events.append({
                    "timestamp": timestamp.isoformat(),
                    "source": "AUTH_LOG",
                    "event_type": "LOGIN_FAILED",
                    "description": f"Failed login: {user} from {ip}",
                    "details": {"user": user, "ip": ip}
                })
    return events

def parse_process_log(log_file):
    """Parse process execution log"""
    events = []
    if not os.path.exists(log_file):
        return events
    
    with open(log_file, "r") as f:
        data = json.load(f)
    
    for e in data:
        try:
            ts = datetime.fromisoformat(e["timestamp"])
            events.append({
                "timestamp": ts.isoformat(),
                "source": "PROCESS_LOG",
                "event_type": "PROCESS_START",
                "description": f"Process started: {e['name']} (PID: {e['pid']})",
                "details": {
                    "pid": e["pid"], "ppid": e["ppid"],
                    "name": e["name"], "cmdline": e["cmdline"],
                    "user": e["user"], "integrity": e["integrity"]
                }
            })
        except Exception:
            pass
    return events

def parse_usb_log(log_file):
    """Parse USB activity log"""
    events = []
    if not os.path.exists(log_file):
        return events
    
    with open(log_file, "r") as f:
        data = json.load(f)
    
    for e in data:
        try:
            ts = datetime.fromisoformat(e["timestamp"])
            action = "DEVICE_CONNECTED" if e["action"] == "DEVICE_ARRIVAL" else "DEVICE_DISCONNECTED"
            events.append({
                "timestamp": ts.isoformat(),
                "source": "USB_LOG",
                "event_type": action,
                "description": f"USB {action.lower()}: {e['description']}",
                "details": {
                    "device_id": e["device_id"], "vid": e["vid"],
                    "pid": e["pid"], "serial": e.get("serial", ""),
                    "drive_letter": e.get("drive_letter", ""), "user": e["user"]
                }
            })
        except Exception:
            pass
    return events

def create_sample_logs():
    """Create sample log files for demonstration"""
    # auth.log
    with open("auth.log", "w") as f:
        base = datetime.now() - timedelta(hours=5)
        f.write(f"Aug 20 08:00:00 sshd[100]: Accepted password for admin from 192.168.1.50\n")
        f.write(f"Aug 20 08:30:00 sshd[101]: Failed password for root from 10.0.0.100\n")
        f.write(f"Aug 20 08:30:05 sshd[102]: Failed password for root from 10.0.0.100\n")
        f.write(f"Aug 20 08:30:10 sshd[103]: Failed password for admin from 10.0.0.100\n")
        f.write(f"Aug 20 09:00:00 sshd[104]: Accepted password for user from 192.168.1.50\n")
    
    # process_execution_log.json
    now = datetime.now()
    base = now - timedelta(hours=3)
    proc_log = [
        {"timestamp": (base + timedelta(minutes=5)).isoformat(), "pid": 1000, "ppid": 1, "name": "explorer.exe", "cmdline": "C:\\Windows\\explorer.exe", "user": "user", "integrity": "Medium"},
        {"timestamp": (base + timedelta(minutes=10)).isoformat(), "pid": 2000, "ppid": 1000, "name": "powershell.exe", "cmdline": "powershell.exe -enc VwByAGkAdABlAC0ASABvAHMAdAAgACIASABlAGwAbABvACIA", "user": "user", "integrity": "Medium"},
        {"timestamp": (base + timedelta(minutes=15)).isoformat(), "pid": 3000, "ppid": 2000, "name": "cmd.exe", "cmdline": "cmd.exe /c whoami", "user": "user", "integrity": "Medium"},
    ]
    with open("process_execution_log.json", "w") as f:
        json.dump(proc_log, f, indent=2)
    
    # usb_activity_log.json
    usb_log = [
        {"timestamp": (base + timedelta(minutes=20)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "USBSTOR\\DISK&VEN_GENERIC&PROD_FLASH&REV_1.00\\SERIAL123&0",
         "vid": "1234", "pid": "5678", "class": "08", "description": "Generic Flash Drive",
         "serial": "SERIAL123", "drive_letter": "E:", "user": "user"},
        {"timestamp": (base + timedelta(minutes=40)).isoformat(), "event_id": 2006, "action": "DEVICE_REMOVAL",
         "device_id": "USBSTOR\\DISK&VEN_GENERIC&PROD_FLASH&REV_1.00\\SERIAL123&0",
         "drive_letter": "E:", "user": "user"},
    ]
    with open("usb_activity_log.json", "w") as f:
        json.dump(usb_log, f, indent=2)

def generate_timeline(directory=".", hours_back=24, output_json="timeline.json", output_txt="timeline.txt"):
    """Generate consolidated timeline from all sources"""
    cutoff = datetime.now() - timedelta(hours=hours_back)
    all_events = []
    
    # Collect from all sources
    print("Collecting file timestamps...")
    all_events.extend(collect_file_timestamps(directory))
    
    print("Parsing auth log...")
    all_events.extend(parse_auth_log("auth.log"))
    
    print("Parsing process log...")
    all_events.extend(parse_process_log("process_execution_log.json"))
    
    print("Parsing USB log...")
    all_events.extend(parse_usb_log("usb_activity_log.json"))
    
    # Filter by time
    filtered = []
    for e in all_events:
        try:
            ts = datetime.fromisoformat(e["timestamp"])
            if ts >= cutoff:
                filtered.append(e)
        except Exception:
            pass
    
    # Sort chronologically
    filtered.sort(key=lambda x: x["timestamp"])
    
    # Export JSON
    with open(output_json, "w") as f:
        json.dump(filtered, f, indent=2)
    
    # Export human-readable text
    with open(output_txt, "w") as f:
        f.write("="*80 + "\n")
        f.write(f"DIGITAL EVENT TIMELINE (Last {hours_back} hours)\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        
        current_date = None
        for e in filtered:
            ts = datetime.fromisoformat(e["timestamp"])
            date_str = ts.strftime("%Y-%m-%d")
            time_str = ts.strftime("%H:%M:%S")
            
            if date_str != current_date:
                current_date = date_str
                f.write(f"\n--- {date_str} ---\n\n")
            
            f.write(f"[{time_str}] [{e['source']}] {e['event_type']}: {e['description']}\n")
            if e["details"]:
                for k, v in e["details"].items():
                    if v:
                        f.write(f"    {k}: {v}\n")
            f.write("\n")
    
    print(f"\nTimeline generated: {len(filtered)} events")
    print(f"  JSON: {output_json}")
    print(f"  Text: {output_txt}")
    
    return filtered

def main():
    create_sample_logs()
    print("Generating digital event timeline...")
    timeline = generate_timeline(".", hours_back=24)
    
    # Print summary
    print(f"\n{'='*60}")
    print("TIMELINE SUMMARY")
    print(f"{'='*60}")
    source_counts = defaultdict(int)
    type_counts = defaultdict(int)
    for e in timeline:
        source_counts[e["source"]] += 1
        type_counts[e["event_type"]] += 1
    
    print("\nBy Source:")
    for src, count in sorted(source_counts.items(), key=lambda x: -x[1]):
        print(f"  {src}: {count}")
    
    print("\nBy Event Type:")
    for etype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {etype}: {count}")

if __name__ == "__main__":
    main()

Generating digital event timeline...
Parsing auth log...
Parsing process log...
Parsing USB log...

Timeline generated: 172 events
  JSON: timeline.json
  Text: timeline.txt

TIMELINE SUMMARY

By Source:
  FILESYSTEM: 163
  AUTH_LOG: 5
  PROCESS_LOG: 3
  USB_LOG: 1

By Event Type:
  FILE_ACCESSED: 61
  FILE_MODIFIED: 51
  FILE_CREATED: 51
  PROCESS_START: 3
  LOGIN_FAILED: 3
  LOGIN_SUCCESS: 2
  DEVICE_CONNECTED: 1


## **Result**
This the program successfully generates a digital event timeline using file timestamps and system log records.